# 1. Model Training Notebook

This notebook mirrors the full training workflow of `MILK10k_model.ipynb`, adapted to the final datasets and metadata decisions obtained from the EDA.

The training pipeline will use the final exported datasets `dataset_11f` and `dataset_2f`.


## 2. Global Imports

All core imports are centralized here.


In [ ]:
import copy
import gc
import os
import random
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm


## 3. Global Configuration

This cell defines the Experiment 1 search space, the output folders, and the full hyperparameter grid that will be executed.


In [ ]:
def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / "src" / "milk10k").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root. Launch the notebook from inside the project.")


PROJECT_ROOT = find_project_root()

MODELS_TO_RUN = ["Resnet18", "ViT"]
TASKS_TO_RUN = ["11f", "2f"]
SENSORS_TO_RUN = ["clinical", "dermoscopic"]

LRS_BY_MODEL = {
    "Resnet18": [1e-4],
    "ViT": [1e-5, 3e-5],
}

BATCH_SIZES_BY_TASK = {
    "11f": [32, 64, 128],
    "2f": [16, 32, 64, 128],
}

# Final configurations reported in the thesis after validation FP-score selection.
# The training loop below still evaluates the full LR x batch-size grid.
# SELECTED_RUN_CONFIGS = [
#     {"task": "11f", "sensor": "dermoscopic", "model": "ViT", "lr": 3e-5, "batch_size": 32},
#     {"task": "11f", "sensor": "clinical", "model": "ViT", "lr": 3e-5, "batch_size": 64},
#     {"task": "2f", "sensor": "dermoscopic", "model": "ViT", "lr": 3e-5, "batch_size": 16},
#     {"task": "2f", "sensor": "clinical", "model": "ViT", "lr": 1e-5, "batch_size": 16},
# ]

EXPERIMENT_CONFIGS = []
for model_name in MODELS_TO_RUN:
    for task in TASKS_TO_RUN:
        for sensor in SENSORS_TO_RUN:
            for lr in LRS_BY_MODEL[model_name]:
                for batch_size in BATCH_SIZES_BY_TASK[task]:
                    EXPERIMENT_CONFIGS.append(
                        {
                            "task": task,
                            "sensor": sensor,
                            "model": model_name,
                            "lr": lr,
                            "batch_size": batch_size,
                        }
                    )

EARLY_STOPPING_PATIENCE_BY_MODEL = {
    "Resnet18": 5,
    "ViT": 3,
}

N_RUNS_BY_MODEL = {
    "Resnet18": 10,
    "ViT": 5,
}

EPOCHS = 100
ALPHA = 0.40
BASE_SEED = 42
RESUME = True

NUM_WORKERS = min(4, os.cpu_count() or 1)
PIN_MEMORY = torch.cuda.is_available()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXPERIMENT_ROOT = PROJECT_ROOT / "Experiment_1" / "training_runs_final_fixed"
MODELS_ROOT = EXPERIMENT_ROOT / "models"
REPORTS_ROOT = EXPERIMENT_ROOT / "reports"
RUNS_ROOT = EXPERIMENT_ROOT / "runs"

for path in [MODELS_ROOT, REPORTS_ROOT, RUNS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_ROOTS = {
    ("11f", "clinical"): PROJECT_ROOT / "data" / "dataset_11f" / "clinical",
    ("11f", "dermoscopic"): PROJECT_ROOT / "data" / "dataset_11f" / "dermoscopic",
    ("2f", "clinical"): PROJECT_ROOT / "data" / "dataset_2f" / "clinical",
    ("2f", "dermoscopic"): PROJECT_ROOT / "data" / "dataset_2f" / "dermoscopic",
}

EXPECTED_CLASSES = {
    "11f": 11,
    "2f": 2,
}

print("Device:", DEVICE)
print("Experiment root:", EXPERIMENT_ROOT)
print("Alpha for objective:", ALPHA)
print(f"Total Experiment 1 configurations: {len(EXPERIMENT_CONFIGS)}")
print("\nFull Experiment 1 grid:")
for config in EXPERIMENT_CONFIGS:
    print(
        f"- task={config['task']} | sensor={config['sensor']} | model={config['model']} | "
        f"lr={config['lr']} | batch_size={config['batch_size']}"
    )

print("\nEarly stopping patience by model:")
for model_name, patience in EARLY_STOPPING_PATIENCE_BY_MODEL.items():
    print(f"- {model_name}: {patience}")


## 4. Metadata Preparation

This section loads the training metadata and prepares the subgroup variables that will later be used for evaluation and fairness analysis.

The metadata are indexed by `isic_id` and reduced to the variables used later in the notebook: `sex`, `skin_tone`, and `age`.

The `sex` variable is kept simple because the metadata only contain `male` and `female`, with no missing values. For `age`, missing values are assigned to `unknown`, since `age_approx` does contain missing values in the metadata.


In [ ]:
metadata_path = Path("data/raw/MILK10k_Training_Metadata.csv")
raw_metadata_df = pd.read_csv(
    metadata_path,
    usecols=["isic_id", "skin_tone_class", "sex", "age_approx"],
).copy()

metadata_df = pd.DataFrame({
    "isic_id": raw_metadata_df["isic_id"],
    "sex": raw_metadata_df["sex"].fillna("unknown").astype(str).str.strip().str.lower(),
    "skin_tone": raw_metadata_df["skin_tone_class"].fillna(-1).astype(int),
})

metadata_df["age"] = raw_metadata_df["age_approx"].apply(
    lambda value: "unknown" if pd.isna(value) else int(value)
)

SEX_TO_ID = {"male": 0, "female": 1, "unknown": 2}

isic_to_skin = dict(zip(metadata_df["isic_id"], metadata_df["skin_tone"]))
isic_to_sex = dict(zip(metadata_df["isic_id"], metadata_df["sex"]))
isic_to_age = dict(zip(metadata_df["isic_id"], metadata_df["age"]))


def normalize_sex(value: str) -> str:
    value = (value or "unknown").strip().lower()
    return value if value in SEX_TO_ID else "unknown"


def normalize_age(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "unknown"
    if isinstance(value, str):
        stripped = value.strip().lower()
        if stripped == "unknown" or stripped == "":
            return "unknown"
        try:
            return int(float(stripped))
        except Exception:
            return "unknown"
    try:
        return int(value)
    except Exception:
        return "unknown"


def format_lr(lr: float) -> str:
    return f"{lr:.0e}".replace("e-0", "e-").replace("e+0", "e+")


def safe_name(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(name).strip().lower()).strip("_")


print("metadata_df:", metadata_df.shape)
metadata_df.head()


## 5. Transforms and Dataset Loading

This section defines the preprocessing pipeline and the helper functions used to load `train`, `validation`, and `test` datasets from the exported folders.


In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


IMAGE_SIZE = 224

DATA_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
])


def load_datasets(task: str, sensor: str):
    dataset_root = DATASET_ROOTS[(task, sensor)]

    train_dataset = ImageFolder(dataset_root / "train", transform=DATA_TRANSFORM)
    val_dataset = ImageFolder(dataset_root / "validation", transform=DATA_TRANSFORM)
    test_dataset = ImageFolder(dataset_root / "test", transform=DATA_TRANSFORM)

    class_names = train_dataset.classes
    if len(class_names) != EXPECTED_CLASSES[task]:
        raise RuntimeError(
            f"Expected {EXPECTED_CLASSES[task]} classes for {task}, but found {len(class_names)} in {dataset_root}."
        )

    if val_dataset.classes != class_names or test_dataset.classes != class_names:
        raise RuntimeError("Class folders are not consistent across train, validation, and test.")

    return train_dataset, val_dataset, test_dataset, class_names


def make_dataloaders(train_dataset, val_dataset, test_dataset, batch_size: int):
    persistent = NUM_WORKERS > 0

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=persistent,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=persistent,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=persistent,
    )

    return train_loader, val_loader, test_loader


set_all_seeds(BASE_SEED)
sample_task = TASKS_TO_RUN[0]
sample_sensor = SENSORS_TO_RUN[0]
sample_train_dataset, sample_val_dataset, sample_test_dataset, sample_class_names = load_datasets(sample_task, sample_sensor)

print(f"Sample dataset: task={sample_task}, sensor={sample_sensor}")
print(f"- train: {len(sample_train_dataset)} images")
print(f"- validation: {len(sample_val_dataset)} images")
print(f"- test: {len(sample_test_dataset)} images")
print(f"- classes ({len(sample_class_names)}): {sample_class_names}")


## 6. Model Factory

This section defines the model-construction helpers used in the experiments.


In [ ]:
def cleanup_memory(*objs):
    for obj in objs:
        try:
            if hasattr(obj, "to"):
                obj.to("cpu")
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def create_model(model_name: str, num_classes: int):
    if model_name == "Resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )
        return model.to(DEVICE)

    if model_name == "ViT":
        try:
            import timm
        except Exception as error:
            raise ImportError("timm is required for ViT. Install it before running this notebook.") from error

        model = timm.create_model(
            "vit_base_patch32_224",
            pretrained=True,
            num_classes=num_classes,
        )
        return model.to(DEVICE)

    raise ValueError(f"Unsupported model: {model_name}")


sample_model_name = MODELS_TO_RUN[0]
sample_num_classes = EXPECTED_CLASSES[TASKS_TO_RUN[0]]
sample_model = create_model(sample_model_name, sample_num_classes)

print(f"Sample model: {sample_model_name}")
print(f"Number of classes: {sample_num_classes}")
print(f"Device: {DEVICE}")


## 7. Training and Validation

This section defines the shared training, evaluation, fairness, and reporting helpers used across all experiments. The fairness component uses Equalized Odds Gap, computed from both true positive rate and false positive rate disparities across sensitive groups. The evaluation output also stores one-vs-rest per-class TPR and FPR values, together with their split-level averages. Early stopping and checkpoint selection are based on validation AUC.


In [ ]:
def compute_f1(y_true, y_pred, num_classes: int) -> float:
    if num_classes == 2:
        return float(f1_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0))
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


def compute_auc(y_true, y_prob, num_classes: int) -> float:
    try:
        if num_classes == 2:
            return float(roc_auc_score(y_true, y_prob[:, 1]))
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
    except Exception:
        return float("nan")


def compute_tpr_fpr_by_class(y_true, y_pred, class_names):
    rows = []
    tpr_values = []
    fpr_values = []

    for idx, class_name in enumerate(class_names):
        y_true_bin = y_true == idx
        y_pred_bin = y_pred == idx

        tp = np.sum(y_true_bin & y_pred_bin)
        fn = np.sum(y_true_bin & (~y_pred_bin))
        fp = np.sum((~y_true_bin) & y_pred_bin)
        tn = np.sum((~y_true_bin) & (~y_pred_bin))

        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

        rows.append({
            "class": class_name,
            "tpr": float(tpr) if not np.isnan(tpr) else np.nan,
            "fpr": float(fpr) if not np.isnan(fpr) else np.nan,
        })
        tpr_values.append(tpr)
        fpr_values.append(fpr)

    summary = {
        "avg_tpr": float(np.nanmean(tpr_values)) if len(tpr_values) else np.nan,
        "avg_fpr": float(np.nanmean(fpr_values)) if len(fpr_values) else np.nan,
    }
    return pd.DataFrame(rows), summary


def compute_eo_gap(y_true, y_pred, sensitive, num_classes: int, ignore_values=None):
    """Compute a multiclass Equalized Odds Gap.

    For each class and sensitive group, the function computes both TPR and FPR.
    The class-level gap is defined as the mean absolute deviation from the group
    mean across TPR and FPR, and the final score averages these gaps across classes.

    The function name is kept for backward compatibility with the rest of the notebook,
    but the stored value now corresponds to Equalized Odds Gap rather than the previous
    Equal Opportunity-only implementation.
    """
    sensitive = np.asarray(sensitive)
    if ignore_values is not None:
        ignore_values = set(ignore_values)
        keep = np.asarray([value not in ignore_values for value in sensitive], dtype=bool)
        y_true = y_true[keep]
        y_pred = y_pred[keep]
        sensitive = sensitive[keep]

    groups = sorted(set(sensitive.tolist()))
    if len(groups) <= 1:
        return 0.0

    class_gaps = []
    for class_idx in range(num_classes):
        tprs = []
        fprs = []

        for group in groups:
            idx = sensitive == group
            yt = y_true[idx] == class_idx
            yp = y_pred[idx] == class_idx

            tp = np.sum(yt & yp)
            fn = np.sum(yt & (~yp))
            fp = np.sum((~yt) & yp)
            tn = np.sum((~yt) & (~yp))

            tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
            tprs.append(tpr)
            fprs.append(fpr)

        mean_tpr = np.nanmean(tprs)
        mean_fpr = np.nanmean(fprs)
        mean_tpr = 0.0 if np.isnan(mean_tpr) else mean_tpr
        mean_fpr = 0.0 if np.isnan(mean_fpr) else mean_fpr

        tpr_gaps = [abs((mean_tpr if np.isnan(tpr) else tpr) - mean_tpr) for tpr in tprs]
        fpr_gaps = [abs((mean_fpr if np.isnan(fpr) else fpr) - mean_fpr) for fpr in fprs]

        class_gap = 0.5 * (float(np.mean(tpr_gaps)) + float(np.mean(fpr_gaps)))
        class_gaps.append(class_gap)

    return float(np.mean(class_gaps)) if class_gaps else 0.0


def compute_G(eo_gap_age: float, eo_gap_sex: float, eo_gap_skin: float) -> float:
    return float(np.mean([
        1 - eo_gap_age,
        1 - eo_gap_sex,
        1 - eo_gap_skin,
    ]))


def compute_objective(auc: float, g_value: float, alpha: float = ALPHA) -> float:
    return float(alpha * auc + (1 - alpha) * g_value)


def get_predictions_targets_probs(model, loader):
    model.eval()
    y_true_all, y_pred_all, y_prob_all = [], [], []
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE)
            logits = model(images)
            y_true_all.append(labels.numpy())
            y_pred_all.append(logits.argmax(dim=1).cpu().numpy())
            y_prob_all.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(y_true_all), np.concatenate(y_pred_all), np.concatenate(y_prob_all)


def age_sort_key(value):
    if value == "unknown":
        return (1, float("inf"))
    return (0, int(value))


def build_sensitive_arrays(dataset):
    skin_vals, sex_vals, age_vals = [], [], []
    for path, _ in dataset.samples:
        isic_id = Path(path).stem
        skin_vals.append(int(isic_to_skin.get(isic_id, -1)))
        sex_vals.append(SEX_TO_ID[normalize_sex(isic_to_sex.get(isic_id, "unknown"))])
        age_vals.append(normalize_age(isic_to_age.get(isic_id, "unknown")))

    unique_age_groups = sorted(set(age_vals), key=age_sort_key)
    age_map = {value: idx for idx, value in enumerate(unique_age_groups)}
    age_ids = np.array([age_map[value] for value in age_vals], dtype=np.int32)

    return np.array(skin_vals, dtype=np.int32), np.array(sex_vals, dtype=np.int32), age_ids


def evaluate_split(model, loader, dataset, class_names):
    y_true, y_pred, y_prob = get_predictions_targets_probs(model, loader)
    num_classes = len(class_names)
    skin_arr, sex_arr, age_arr = build_sensitive_arrays(dataset)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": compute_f1(y_true, y_pred, num_classes),
        "auc": compute_auc(y_true, y_prob, num_classes),
    }
    metrics["eo_gap_skin"] = compute_eo_gap(y_true, y_pred, skin_arr, num_classes, ignore_values={0, -1})
    metrics["eo_gap_sex"] = compute_eo_gap(y_true, y_pred, sex_arr, num_classes)
    metrics["eo_gap_age"] = compute_eo_gap(y_true, y_pred, age_arr, num_classes)
    metrics["G"] = compute_G(
        metrics["eo_gap_age"],
        metrics["eo_gap_sex"],
        metrics["eo_gap_skin"],
    )
    metrics["objective"] = compute_objective(metrics["auc"], metrics["G"], ALPHA)

    tpr_fpr_df, tpr_fpr_summary = compute_tpr_fpr_by_class(y_true, y_pred, class_names)
    metrics["avg_tpr"] = tpr_fpr_summary["avg_tpr"]
    metrics["avg_fpr"] = tpr_fpr_summary["avg_fpr"]

    per_class_rows = []
    for idx, class_name in enumerate(class_names):
        mask = y_true == idx
        class_acc = float(accuracy_score(y_true[mask], y_pred[mask])) if np.sum(mask) else np.nan
        per_class_rows.append({"class": class_name, "n": int(np.sum(mask)), "accuracy": class_acc})
        metrics[f"acc_{safe_name(class_name)}"] = class_acc

    per_class_df = pd.DataFrame(per_class_rows).merge(tpr_fpr_df, on="class", how="left")

    for _, row in tpr_fpr_df.iterrows():
        class_key = safe_name(row["class"])
        metrics[f"tpr_{class_key}"] = row["tpr"]
        metrics[f"fpr_{class_key}"] = row["fpr"]

    return metrics, per_class_df


def train_step(model, dataloader, loss_fn, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / max(total, 1), correct / max(total, 1)


def val_step(model, dataloader, loss_fn, num_classes: int):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    y_true_all = []
    y_prob_all = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss = loss_fn(logits, labels)

            running_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            y_true_all.append(labels.cpu().numpy())
            y_prob_all.append(torch.softmax(logits, dim=1).cpu().numpy())

    y_true = np.concatenate(y_true_all) if y_true_all else np.array([])
    y_prob = np.concatenate(y_prob_all) if y_prob_all else np.array([])
    val_auc = compute_auc(y_true, y_prob, num_classes) if y_true.size else float("nan")

    return running_loss / max(total, 1), correct / max(total, 1), val_auc


def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs, patience, num_classes: int):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_auc": []}
    best_val_auc = -np.inf
    best_state = copy.deepcopy(model.state_dict())
    patience_count = 0

    for epoch in tqdm(range(epochs), leave=False):
        train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc, val_auc = val_step(model, val_loader, loss_fn, num_classes)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_auc={val_auc:.4f}"
        )

        improved = val_auc > best_val_auc if not np.isnan(val_auc) else False
        if improved:
            print(" Validation AUC improved, saving best model...")
            best_val_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    model.load_state_dict(best_state)
    return history


def append_row(csv_path, row):
    old_df = read_csv_if_exists(csv_path)
    new_df = pd.DataFrame([row])
    if old_df.empty:
        out_df = new_df
    else:
        cols = list(dict.fromkeys(list(old_df.columns) + list(new_df.columns)))
        out_df = pd.concat([old_df.reindex(columns=cols), new_df.reindex(columns=cols)], ignore_index=True)
    out_df.to_csv(csv_path, index=False, lineterminator="\n")


def read_csv_if_exists(csv_path):
    if not csv_path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()


def compute_averages_from_experiments(exp_df):
    if exp_df.empty:
        return pd.DataFrame()

    ok_df = exp_df[exp_df["status"] == "ok"].copy()
    if ok_df.empty:
        return pd.DataFrame()

    group_cols = ["model", "task", "sensor", "hp_tag", "lr", "batch_size"]
    excluded = {"run_id", "seed", "status", "model_path", "report_path", "best_epoch", "best_val_auc", "error"}
    metric_cols = [col for col in ok_df.columns if col not in group_cols and col not in excluded]

    return ok_df.groupby(group_cols).agg(
        n_runs_completed=("run_id", "count"),
        **{col: (col, "mean") for col in metric_cols},
    ).reset_index()


def df_to_md(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)


def write_run_report(path, run_id, model_name, task, sensor, lr, batch_size, seed, best_epoch, best_val_auc, model_path, val_metrics, test_metrics, val_class_df, test_class_df):
    global_df = pd.DataFrame([
        {
            "Split": "Validation",
            "Accuracy": val_metrics["accuracy"],
            "F1": val_metrics["f1"],
            "AUC": val_metrics["auc"],
            "Average TPR": val_metrics["avg_tpr"],
            "Average FPR": val_metrics["avg_fpr"],
            "G": val_metrics["G"],
            "Objective": val_metrics["objective"],
            "Equalized Odds Gap (skin)": val_metrics["eo_gap_skin"],
            "Equalized Odds Gap (sex)": val_metrics["eo_gap_sex"],
            "Equalized Odds Gap (age)": val_metrics["eo_gap_age"],
        },
        {
            "Split": "Test",
            "Accuracy": test_metrics["accuracy"],
            "F1": test_metrics["f1"],
            "AUC": test_metrics["auc"],
            "Average TPR": test_metrics["avg_tpr"],
            "Average FPR": test_metrics["avg_fpr"],
            "G": test_metrics["G"],
            "Objective": test_metrics["objective"],
            "Equalized Odds Gap (skin)": test_metrics["eo_gap_skin"],
            "Equalized Odds Gap (sex)": test_metrics["eo_gap_sex"],
            "Equalized Odds Gap (age)": test_metrics["eo_gap_age"],
        },
    ])

    text = f"""# Run {run_id}

## Hyperparameters
- **model**: {model_name}
- **task**: {task}
- **sensor**: {sensor}
- **learning_rate**: {lr}
- **batch_size**: {batch_size}
- **alpha**: {ALPHA}
- **seed**: {seed}
- **best_epoch**: {best_epoch}
- **best_val_auc**: {best_val_auc:.6f}
- **model_path**: {model_path}

## Global metrics
{df_to_md(global_df)}

## Validation per-class metrics
{df_to_md(val_class_df)}

## Test per-class metrics
{df_to_md(test_class_df)}
"""
    path.write_text(text, encoding="utf-8")


def write_summary_report(path, model_name, task, sensor, hp_tag, lr, batch_size, avg_row):
    metric_rows = []
    for metric in ["accuracy", "f1", "auc", "avg_tpr", "avg_fpr", "G", "objective", "eo_gap_skin", "eo_gap_sex", "eo_gap_age"]:
        metric_rows.append({
            "Metric": metric,
            "Validation Mean": float(avg_row.get(f"val_{metric}", np.nan)),
            "Test Mean": float(avg_row.get(f"test_{metric}", np.nan)),
        })

    class_keys = sorted({
        col.replace("val_acc_", "", 1)
        for col in avg_row.index
        if col.startswith("val_acc_")
    })

    val_class_rows = [
        {
            "class": class_key,
            "accuracy_mean": float(avg_row.get(f"val_acc_{class_key}", np.nan)),
            "tpr_mean": float(avg_row.get(f"val_tpr_{class_key}", np.nan)),
            "fpr_mean": float(avg_row.get(f"val_fpr_{class_key}", np.nan)),
        }
        for class_key in class_keys
    ]
    test_class_rows = [
        {
            "class": class_key,
            "accuracy_mean": float(avg_row.get(f"test_acc_{class_key}", np.nan)),
            "tpr_mean": float(avg_row.get(f"test_tpr_{class_key}", np.nan)),
            "fpr_mean": float(avg_row.get(f"test_fpr_{class_key}", np.nan)),
        }
        for class_key in class_keys
    ]

    text = f"""# Summary {hp_tag}

## Hyperparameters
- **model**: {model_name}
- **task**: {task}
- **sensor**: {sensor}
- **learning_rate**: {lr}
- **batch_size**: {batch_size}
- **alpha**: {ALPHA}
- **n_runs_completed**: {int(avg_row['n_runs_completed'])}

## Global metrics (mean)
{df_to_md(pd.DataFrame(metric_rows))}

## Validation per-class metrics (mean)
{df_to_md(pd.DataFrame(val_class_rows))}

## Test per-class metrics (mean)
{df_to_md(pd.DataFrame(test_class_rows))}
"""
    path.write_text(text, encoding="utf-8")


## 8. Full Training Loop

This section runs the full Experiment 1 grid defined in the global configuration cell and stores models, reports, and intermediate CSV summaries directly during training.


In [ ]:
for config in EXPERIMENT_CONFIGS:
    task = config["task"]
    sensor = config["sensor"]
    model_name = config["model"]
    lr = config["lr"]
    batch_size = config["batch_size"]

    if (task, sensor) not in DATASET_ROOTS:
        continue

    train_data, val_data, test_data, class_names = load_datasets(task, sensor)
    n_runs = N_RUNS_BY_MODEL[model_name]

    combo_models_root = MODELS_ROOT / model_name / task / sensor
    combo_reports_root = REPORTS_ROOT / model_name / task / sensor
    combo_runs_root = RUNS_ROOT / model_name / task / sensor
    for path in [combo_models_root, combo_reports_root, combo_runs_root]:
        path.mkdir(parents=True, exist_ok=True)

    experiments_csv = combo_runs_root / "experiments.csv"
    averages_csv = combo_runs_root / "averages.csv"

    print("\n" + "=" * 80)
    print(f"{model_name} | {task} | {sensor} | lr={lr} | batch_size={batch_size} | runs={n_runs}")
    print("=" * 80)

    hp_tag = f"lr{format_lr(lr)}_bs{batch_size}"
    hp_model_dir = combo_models_root / hp_tag
    hp_report_dir = combo_reports_root / hp_tag
    hp_model_dir.mkdir(parents=True, exist_ok=True)
    hp_report_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, test_loader = make_dataloaders(
        train_data,
        val_data,
        test_data,
        batch_size=batch_size,
    )
    print(f"\n--- {hp_tag} ---")

    for run_idx in range(1, n_runs + 1):
        run_seed = BASE_SEED + run_idx
        run_id = f"{model_name}_{task}_{sensor}_{hp_tag}_run{run_idx:02d}"
        model_path = hp_model_dir / f"{run_id}.pth"
        report_path = hp_report_dir / f"{run_id}.md"

        exp_df = read_csv_if_exists(experiments_csv)
        if RESUME and not exp_df.empty and "run_id" in exp_df.columns:
            already_ok = ((exp_df["run_id"] == run_id) & (exp_df["status"] == "ok")).any()
            if already_ok and model_path.exists():
                print(f"Skipping: {run_id}")
                continue

        set_all_seeds(run_seed)
        model = None
        try:
            model = create_model(model_name, num_classes=len(class_names))
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
            loss_fn = nn.CrossEntropyLoss()

            history = train_model(
                model,
                train_loader,
                val_loader,
                optimizer,
                loss_fn,
                EPOCHS,
                EARLY_STOPPING_PATIENCE_BY_MODEL[model_name],
                num_classes=len(class_names),
            )
            val_metrics, val_class_df = evaluate_split(model, val_loader, val_data, class_names)
            test_metrics, test_class_df = evaluate_split(model, test_loader, test_data, class_names)

            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "model_name": model_name,
                    "task": task,
                    "sensor": sensor,
                    "num_classes": len(class_names),
                    "class_names": class_names,
                    "lr": lr,
                    "batch_size": batch_size,
                    "run_id": run_id,
                },
                model_path,
            )

            best_epoch = int(np.nanargmax(history["val_auc"])) + 1 if history["val_auc"] else 0
            best_val_auc = float(np.nanmax(history["val_auc"])) if history["val_auc"] else np.nan

            row = {
                "model": model_name,
                "task": task,
                "sensor": sensor,
                "hp_tag": hp_tag,
                "run_id": run_id,
                "seed": run_seed,
                "lr": lr,
                "batch_size": batch_size,
                "status": "ok",
                "model_path": str(model_path),
                "report_path": str(report_path),
                "best_epoch": best_epoch,
                "best_val_auc": best_val_auc,
                "val_accuracy": val_metrics["accuracy"],
                "val_f1": val_metrics["f1"],
                "val_auc": val_metrics["auc"],
                "val_avg_tpr": val_metrics["avg_tpr"],
                "val_avg_fpr": val_metrics["avg_fpr"],
                "val_G": val_metrics["G"],
                "val_objective": val_metrics["objective"],
                "val_eo_gap_skin": val_metrics["eo_gap_skin"],
                "val_eo_gap_sex": val_metrics["eo_gap_sex"],
                "val_eo_gap_age": val_metrics["eo_gap_age"],
                "test_accuracy": test_metrics["accuracy"],
                "test_f1": test_metrics["f1"],
                "test_auc": test_metrics["auc"],
                "test_avg_tpr": test_metrics["avg_tpr"],
                "test_avg_fpr": test_metrics["avg_fpr"],
                "test_G": test_metrics["G"],
                "test_objective": test_metrics["objective"],
                "test_eo_gap_skin": test_metrics["eo_gap_skin"],
                "test_eo_gap_sex": test_metrics["eo_gap_sex"],
                "test_eo_gap_age": test_metrics["eo_gap_age"],
            }
            for key, value in val_metrics.items():
                if key.startswith("acc_") or key.startswith("tpr_") or key.startswith("fpr_"):
                    row[f"val_{key}"] = value
            for key, value in test_metrics.items():
                if key.startswith("acc_") or key.startswith("tpr_") or key.startswith("fpr_"):
                    row[f"test_{key}"] = value

            append_row(experiments_csv, row)
            write_run_report(
                report_path,
                run_id,
                model_name,
                task,
                sensor,
                lr,
                batch_size,
                run_seed,
                best_epoch,
                best_val_auc,
                model_path,
                val_metrics,
                test_metrics,
                val_class_df,
                test_class_df,
            )
            print(
                f"run {run_idx:02d}/{n_runs} | "
                f"val_acc={val_metrics['accuracy']:.4f} "
                f"test_acc={test_metrics['accuracy']:.4f} "
                f"test_auc={test_metrics['auc']:.4f} "
                f"test_G={test_metrics['G']:.4f} "
                f"test_objective={test_metrics['objective']:.4f}"
            )
        except Exception as error:
            append_row(
                experiments_csv,
                {
                    "model": model_name,
                    "task": task,
                    "sensor": sensor,
                    "hp_tag": hp_tag,
                    "run_id": run_id,
                    "seed": run_seed,
                    "lr": lr,
                    "batch_size": batch_size,
                    "status": "fail",
                    "error": f"{type(error).__name__}: {error}",
                },
            )
            print(f"ERROR in {run_id}: {error}")
        finally:
            cleanup_memory(model)

    hp_exp_df = read_csv_if_exists(experiments_csv)
    hp_avg_df = compute_averages_from_experiments(hp_exp_df)
    if not hp_avg_df.empty:
        hp_avg_df.to_csv(averages_csv, index=False)
        hp_row = hp_avg_df[hp_avg_df["hp_tag"] == hp_tag]
        if not hp_row.empty and int(hp_row.iloc[0]["n_runs_completed"]) >= n_runs:
            write_summary_report(
                hp_report_dir / "summary.md",
                model_name,
                task,
                sensor,
                hp_tag,
                lr,
                batch_size,
                hp_row.iloc[0],
            )
            print(f"summary.md updated for {hp_tag}")

    cleanup_memory()

print("Training pipeline finished.")


## 9. Final Consolidation

This section rebuilds the aggregated CSV and markdown summaries for each `(model, task, sensor)` combination.


In [ ]:
def build_top5_table(avg_df, top_k=5):
    if avg_df.empty:
        return pd.DataFrame()

    sort_cols = [col for col in ["test_objective", "test_auc", "val_objective"] if col in avg_df.columns]
    if not sort_cols:
        return pd.DataFrame()

    top_df = avg_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).head(top_k).copy()
    core_cols = [
        "model", "task", "sensor", "hp_tag", "lr", "batch_size", "n_runs_completed",
        "val_accuracy", "val_f1", "val_auc", "val_avg_tpr", "val_avg_fpr", "val_G", "val_objective", "val_eo_gap_skin", "val_eo_gap_sex", "val_eo_gap_age",
        "test_accuracy", "test_f1", "test_auc", "test_avg_tpr", "test_avg_fpr", "test_G", "test_objective", "test_eo_gap_skin", "test_eo_gap_sex", "test_eo_gap_age",
    ]
    class_cols = sorted([
        col for col in top_df.columns
        if col.startswith("val_acc_") or col.startswith("test_acc_") or col.startswith("val_tpr_") or col.startswith("test_tpr_") or col.startswith("val_fpr_") or col.startswith("test_fpr_")
    ])
    cols = [col for col in core_cols + class_cols if col in top_df.columns]
    return top_df[cols]


for model_name in MODELS_TO_RUN:
    for task in TASKS_TO_RUN:
        for sensor in SENSORS_TO_RUN:
            run_dir = RUNS_ROOT / model_name / task / sensor
            if not run_dir.exists():
                continue

            experiments_csv = run_dir / "experiments.csv"
            averages_csv = run_dir / "averages.csv"
            top5_csv = run_dir / "top5_summary.csv"
            top5_md = run_dir / "top5_summary.md"

            exp_df = read_csv_if_exists(experiments_csv)
            if exp_df.empty:
                continue

            avg_df = compute_averages_from_experiments(exp_df)
            if avg_df.empty:
                continue

            avg_df.to_csv(averages_csv, index=False)
            top5_df = build_top5_table(avg_df, top_k=5)
            top5_df.to_csv(top5_csv, index=False)

            md_text = f"""# Top-5 Summary ({model_name} | {task} | {sensor})

- Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- Source: {averages_csv}
- Ranking: test_objective, test_auc, val_objective

{df_to_md(top5_df)}
"""
            top5_md.write_text(md_text, encoding="utf-8")

            print(f"Updated: {run_dir}")

print("Final consolidation complete.")


## 10. Minimal Execution Order

Run the notebook from top to bottom. The full training loop writes intermediate files as it progresses, and the final consolidation cell refreshes the global summaries at the end.


## 11. Heatmaps

This section generates LR x batch size heatmaps from the `averages.csv` files, following the same style used in `Experiment_1/try_3/plots/heatmaps`.


In [ ]:
HEATMAPS_ROOT = EXPERIMENT_ROOT / "plots" / "heatmaps"


def format_heatmap_lr(value):
    return f"{float(value):g}"


def format_heatmap_value(metric_key, value):
    if np.isnan(value):
        return ""
    if "eo_gap" in metric_key or metric_key.endswith("_G") or metric_key.endswith("_objective"):
        return f"{value:.4f}"
    return f"{value:.3f}"


def plot_metric_heatmap(pivot_df, title, out_path, metric_key):
    if pivot_df.empty:
        return False

    pivot_df = pivot_df.sort_index(axis=0).reindex(sorted(pivot_df.columns), axis=1)
    matrix = pivot_df.values.astype(float)
    if np.all(np.isnan(matrix)):
        return False

    fig, ax = plt.subplots(
        figsize=(max(8, 1.15 * len(pivot_df.columns) + 4), max(4.5, 0.85 * len(pivot_df.index) + 3)),
        dpi=180,
    )
    image = ax.imshow(matrix, aspect="auto", origin="lower")

    ax.set_title(title)
    ax.set_xlabel("batch_size")
    ax.set_ylabel("learning_rate")
    ax.set_xticks(np.arange(len(pivot_df.columns)))
    ax.set_yticks(np.arange(len(pivot_df.index)))
    ax.set_xticklabels([str(int(x)) for x in pivot_df.columns], rotation=45, ha="right")
    ax.set_yticklabels([format_heatmap_lr(y) for y in pivot_df.index])

    finite = matrix[np.isfinite(matrix)]
    threshold = float(np.mean(finite)) if finite.size else 0.0

    for row_idx in range(matrix.shape[0]):
        for col_idx in range(matrix.shape[1]):
            value = matrix[row_idx, col_idx]
            if np.isnan(value):
                continue
            ax.text(
                col_idx,
                row_idx,
                format_heatmap_value(metric_key, value),
                ha="center",
                va="center",
                fontsize=11,
                color=("white" if value >= threshold else "black"),
            )

    colorbar = plt.colorbar(image, ax=ax)
    colorbar.ax.set_ylabel("mean")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return True


heatmap_metrics = [
    ("val_accuracy", "Validation Accuracy"),
    ("val_auc", "Validation AUC"),
    ("val_G", "Validation G"),
    ("val_objective", "Validation Objective"),
    ("val_eo_gap_skin", "Validation Equalized Odds Gap (skin)"),
    ("val_eo_gap_sex", "Validation Equalized Odds Gap (sex)"),
    ("val_eo_gap_age", "Validation Equalized Odds Gap (age)"),
    ("test_accuracy", "Test Accuracy"),
    ("test_auc", "Test AUC"),
    ("test_G", "Test G"),
    ("test_objective", "Test Objective"),
    ("test_eo_gap_skin", "Test Equalized Odds Gap (skin)"),
    ("test_eo_gap_sex", "Test Equalized Odds Gap (sex)"),
    ("test_eo_gap_age", "Test Equalized Odds Gap (age)"),
]

saved = 0
skipped = 0

for model_name in MODELS_TO_RUN:
    for task in TASKS_TO_RUN:
        for sensor in SENSORS_TO_RUN:
            avg_path = RUNS_ROOT / model_name / task / sensor / "averages.csv"
            if not avg_path.exists():
                print(f"Missing: {avg_path}")
                continue

            avg_df = pd.read_csv(avg_path)
            if avg_df.empty:
                print(f"Empty: {avg_path}")
                continue

            required_cols = ["lr", "batch_size"]
            missing_required = [col for col in required_cols if col not in avg_df.columns]
            if missing_required:
                print(f"Missing columns in {avg_path}: {missing_required}")
                continue

            avg_df["lr"] = pd.to_numeric(avg_df["lr"], errors="coerce")
            avg_df["batch_size"] = pd.to_numeric(avg_df["batch_size"], errors="coerce")
            avg_df = avg_df.dropna(subset=["lr", "batch_size"]).copy()
            if avg_df.empty:
                print(f"No valid lr/batch_size in: {avg_path}")
                continue

            avg_df["batch_size"] = avg_df["batch_size"].astype(int)

            for metric_key, metric_name in heatmap_metrics:
                if metric_key not in avg_df.columns:
                    skipped += 1
                    continue

                metric_df = avg_df[["lr", "batch_size", metric_key]].copy()
                metric_df[metric_key] = pd.to_numeric(metric_df[metric_key], errors="coerce")
                pivot_df = metric_df.pivot_table(
                    index="lr",
                    columns="batch_size",
                    values=metric_key,
                    aggfunc="mean",
                )

                out_path = HEATMAPS_ROOT / model_name / task / sensor / f"{metric_key}.png"
                title = f"{model_name} | {task} | {sensor} | {metric_name}"
                ok = plot_metric_heatmap(pivot_df, title, out_path, metric_key)

                if ok:
                    saved += 1
                else:
                    skipped += 1

            print(f"Heatmaps done: {model_name} | {task} | {sensor}")

print(f"\nSaved heatmaps: {saved} | skipped: {skipped}")
print(f"Output root: {HEATMAPS_ROOT.resolve()}")


In [ ]:
def build_best_objective_table(model_name: str):
    rows = []

    for task in TASKS_TO_RUN:
        for sensor in SENSORS_TO_RUN:
            averages_csv = RUNS_ROOT / model_name / task / sensor / "averages.csv"
            if not averages_csv.exists():
                continue

            avg_df = read_csv_if_exists(averages_csv)
            if avg_df.empty or "val_objective" not in avg_df.columns:
                continue

            metric_cols = [
                "lr", "batch_size",
                "val_accuracy", "val_auc", "val_eo_gap_skin", "val_eo_gap_sex", "val_eo_gap_age", "val_G", "val_objective",
                "test_accuracy", "test_auc", "test_eo_gap_skin", "test_eo_gap_sex", "test_eo_gap_age", "test_G", "test_objective",
            ]
            for col in metric_cols:
                if col in avg_df.columns:
                    avg_df[col] = pd.to_numeric(avg_df[col], errors="coerce")

            avg_df = avg_df.sort_values(["val_objective", "test_objective", "val_auc"], ascending=[False, False, False])
            best_row = avg_df.iloc[0]

            rows.append({
                "Task": task,
                "Sensor": sensor,
                "Model": model_name,
                "LR": float(best_row["lr"]),
                "Batch Size": int(best_row["batch_size"]),
                "Accuracy": float(best_row["val_accuracy"]),
                "AUC": float(best_row["val_auc"]),
                "Equalized Odds Gap (Skin)": float(best_row["val_eo_gap_skin"]),
                "Equalized Odds Gap (Sex)": float(best_row["val_eo_gap_sex"]),
                "Equalized Odds Gap (Age)": float(best_row["val_eo_gap_age"]),
                "G": float(best_row["val_G"]),
                "Objective": float(best_row["val_objective"]),
                "Metric": "Validation",
            })

            rows.append({
                "Task": task,
                "Sensor": sensor,
                "Model": model_name,
                "LR": float(best_row["lr"]),
                "Batch Size": int(best_row["batch_size"]),
                "Accuracy": float(best_row["test_accuracy"]),
                "AUC": float(best_row["test_auc"]),
                "Equalized Odds Gap (Skin)": float(best_row["test_eo_gap_skin"]),
                "Equalized Odds Gap (Sex)": float(best_row["test_eo_gap_sex"]),
                "Equalized Odds Gap (Age)": float(best_row["test_eo_gap_age"]),
                "G": float(best_row["test_G"]),
                "Objective": float(best_row["test_objective"]),
                "Metric": "Test",
            })

    out_df = pd.DataFrame(rows, columns=[
        "Task", "Sensor", "Model", "LR", "Batch Size", "Accuracy", "AUC",
        "Equalized Odds Gap (Skin)", "Equalized Odds Gap (Sex)", "Equalized Odds Gap (Age)", "G", "Objective", "Metric",
    ])

    if out_df.empty:
        return out_df

    task_order = {task: idx for idx, task in enumerate(TASKS_TO_RUN)}
    sensor_order = {sensor: idx for idx, sensor in enumerate(SENSORS_TO_RUN)}
    metric_order = {"Validation": 0, "Test": 1}

    out_df["_task_order"] = out_df["Task"].map(task_order)
    out_df["_sensor_order"] = out_df["Sensor"].map(sensor_order)
    out_df["_metric_order"] = out_df["Metric"].map(metric_order)
    out_df = out_df.sort_values(["_task_order", "_sensor_order", "_metric_order"]).drop(columns=["_task_order", "_sensor_order", "_metric_order"]).reset_index(drop=True)

    return out_df


resnet18_best_table = build_best_objective_table("Resnet18")
vit_best_table = build_best_objective_table("ViT")

print("Resnet18 best hyperparameter combinations selected by validation objective")
display(resnet18_best_table)

print("ViT best hyperparameter combinations selected by validation objective")
display(vit_best_table)


In [ ]:
def build_best_model_per_task_sensor_table():
    candidate_rows = []

    for model_name in MODELS_TO_RUN:
        for task in TASKS_TO_RUN:
            for sensor in SENSORS_TO_RUN:
                averages_csv = RUNS_ROOT / model_name / task / sensor / "averages.csv"
                if not averages_csv.exists():
                    continue

                avg_df = read_csv_if_exists(averages_csv)
                if avg_df.empty or "val_objective" not in avg_df.columns:
                    continue

                metric_cols = [
                    "lr", "batch_size",
                    "val_accuracy", "val_auc", "val_eo_gap_skin", "val_eo_gap_sex", "val_eo_gap_age", "val_G", "val_objective",
                    "test_accuracy", "test_auc", "test_eo_gap_skin", "test_eo_gap_sex", "test_eo_gap_age", "test_G", "test_objective",
                ]
                for col in metric_cols:
                    if col in avg_df.columns:
                        avg_df[col] = pd.to_numeric(avg_df[col], errors="coerce")

                avg_df = avg_df.sort_values(["val_objective", "test_objective", "val_auc"], ascending=[False, False, False])
                best_row = avg_df.iloc[0]

                candidate_rows.append({
                    "Task": task,
                    "Sensor": sensor,
                    "Model": model_name,
                    "LR": float(best_row["lr"]),
                    "Batch Size": int(best_row["batch_size"]),
                    "val_accuracy": float(best_row["val_accuracy"]),
                    "val_auc": float(best_row["val_auc"]),
                    "val_eo_gap_skin": float(best_row["val_eo_gap_skin"]),
                    "val_eo_gap_sex": float(best_row["val_eo_gap_sex"]),
                    "val_eo_gap_age": float(best_row["val_eo_gap_age"]),
                    "val_G": float(best_row["val_G"]),
                    "val_objective": float(best_row["val_objective"]),
                    "test_accuracy": float(best_row["test_accuracy"]),
                    "test_auc": float(best_row["test_auc"]),
                    "test_eo_gap_skin": float(best_row["test_eo_gap_skin"]),
                    "test_eo_gap_sex": float(best_row["test_eo_gap_sex"]),
                    "test_eo_gap_age": float(best_row["test_eo_gap_age"]),
                    "test_G": float(best_row["test_G"]),
                    "test_objective": float(best_row["test_objective"]),
                })

    candidates_df = pd.DataFrame(candidate_rows)
    if candidates_df.empty:
        return candidates_df

    selected_rows = []
    for (task, sensor), group_df in candidates_df.groupby(["Task", "Sensor"], sort=False):
        best_model_row = group_df.sort_values(["val_objective", "test_objective", "val_auc"], ascending=[False, False, False]).iloc[0]

        selected_rows.append({
            "Task": task,
            "Sensor": sensor,
            "Model": best_model_row["Model"],
            "LR": best_model_row["LR"],
            "Batch Size": int(best_model_row["Batch Size"]),
            "Accuracy": best_model_row["val_accuracy"],
            "AUC": best_model_row["val_auc"],
            "Equalized Odds Gap (Skin)": best_model_row["val_eo_gap_skin"],
            "Equalized Odds Gap (Sex)": best_model_row["val_eo_gap_sex"],
            "Equalized Odds Gap (Age)": best_model_row["val_eo_gap_age"],
            "G": best_model_row["val_G"],
            "Objective": best_model_row["val_objective"],
            "Metric": "Validation",
        })

        selected_rows.append({
            "Task": task,
            "Sensor": sensor,
            "Model": best_model_row["Model"],
            "LR": best_model_row["LR"],
            "Batch Size": int(best_model_row["Batch Size"]),
            "Accuracy": best_model_row["test_accuracy"],
            "AUC": best_model_row["test_auc"],
            "Equalized Odds Gap (Skin)": best_model_row["test_eo_gap_skin"],
            "Equalized Odds Gap (Sex)": best_model_row["test_eo_gap_sex"],
            "Equalized Odds Gap (Age)": best_model_row["test_eo_gap_age"],
            "G": best_model_row["test_G"],
            "Objective": best_model_row["test_objective"],
            "Metric": "Test",
        })

    out_df = pd.DataFrame(selected_rows, columns=[
        "Task", "Sensor", "Model", "LR", "Batch Size", "Accuracy", "AUC",
        "Equalized Odds Gap (Skin)", "Equalized Odds Gap (Sex)", "Equalized Odds Gap (Age)", "G", "Objective", "Metric",
    ])

    task_order = {task: idx for idx, task in enumerate(TASKS_TO_RUN)}
    sensor_order = {sensor: idx for idx, sensor in enumerate(SENSORS_TO_RUN)}
    metric_order = {"Validation": 0, "Test": 1}

    out_df["_task_order"] = out_df["Task"].map(task_order)
    out_df["_sensor_order"] = out_df["Sensor"].map(sensor_order)
    out_df["_metric_order"] = out_df["Metric"].map(metric_order)
    out_df = out_df.sort_values(["_task_order", "_sensor_order", "_metric_order"]).drop(columns=["_task_order", "_sensor_order", "_metric_order"]).reset_index(drop=True)

    return out_df


best_model_per_task_sensor_table = build_best_model_per_task_sensor_table()
print("Best model per task and sensor selected by validation objective")
display(best_model_per_task_sensor_table)


In [ ]:
def build_2f_dermoscopic_model_comparison_table():
    rows = []

    for model_name in ["Resnet18", "ViT"]:
        averages_csv = RUNS_ROOT / model_name / "2f" / "dermoscopic" / "averages.csv"
        if not averages_csv.exists():
            continue

        avg_df = read_csv_if_exists(averages_csv)
        if avg_df.empty or "val_objective" not in avg_df.columns:
            continue

        metric_cols = [
            "lr", "batch_size",
            "val_accuracy", "val_auc", "val_eo_gap_skin", "val_eo_gap_sex", "val_eo_gap_age", "val_G", "val_objective",
            "test_accuracy", "test_auc", "test_eo_gap_skin", "test_eo_gap_sex", "test_eo_gap_age", "test_G", "test_objective",
        ]
        for col in metric_cols:
            if col in avg_df.columns:
                avg_df[col] = pd.to_numeric(avg_df[col], errors="coerce")

        avg_df = avg_df.sort_values(["val_objective", "test_objective", "val_auc"], ascending=[False, False, False])
        best_row = avg_df.iloc[0]

        rows.append({
            "Task": "2f",
            "Sensor": "dermoscopic",
            "Model": model_name,
            "LR": float(best_row["lr"]),
            "Batch Size": int(best_row["batch_size"]),
            "Accuracy": float(best_row["val_accuracy"]),
            "AUC": float(best_row["val_auc"]),
            "Equalized Odds Gap (Skin)": float(best_row["val_eo_gap_skin"]),
            "Equalized Odds Gap (Sex)": float(best_row["val_eo_gap_sex"]),
            "Equalized Odds Gap (Age)": float(best_row["val_eo_gap_age"]),
            "G": float(best_row["val_G"]),
            "Objective": float(best_row["val_objective"]),
            "Metric": "Validation",
        })

        rows.append({
            "Task": "2f",
            "Sensor": "dermoscopic",
            "Model": model_name,
            "LR": float(best_row["lr"]),
            "Batch Size": int(best_row["batch_size"]),
            "Accuracy": float(best_row["test_accuracy"]),
            "AUC": float(best_row["test_auc"]),
            "Equalized Odds Gap (Skin)": float(best_row["test_eo_gap_skin"]),
            "Equalized Odds Gap (Sex)": float(best_row["test_eo_gap_sex"]),
            "Equalized Odds Gap (Age)": float(best_row["test_eo_gap_age"]),
            "G": float(best_row["test_G"]),
            "Objective": float(best_row["test_objective"]),
            "Metric": "Test",
        })

    out_df = pd.DataFrame(rows, columns=[
        "Task", "Sensor", "Model", "LR", "Batch Size", "Accuracy", "AUC",
        "Equalized Odds Gap (Skin)", "Equalized Odds Gap (Sex)", "Equalized Odds Gap (Age)", "G", "Objective", "Metric",
    ])

    if out_df.empty:
        return out_df

    model_order = {"Resnet18": 0, "ViT": 1}
    metric_order = {"Validation": 0, "Test": 1}
    out_df["_model_order"] = out_df["Model"].map(model_order)
    out_df["_metric_order"] = out_df["Metric"].map(metric_order)
    out_df = out_df.sort_values(["_model_order", "_metric_order"]).drop(columns=["_model_order", "_metric_order"]).reset_index(drop=True)

    return out_df


best_2f_dermoscopic_resnet_vs_vit_table = build_2f_dermoscopic_model_comparison_table()
print("Best 2f dermoscopic configurations: Resnet18 vs ViT")
display(best_2f_dermoscopic_resnet_vs_vit_table)


## Updated TPR/FPR Summary Tables

These tables read the saved `averages.csv` files and display the best configurations selected by validation `Objective`. They include average TPR/FPR and a separate per-class TPR/FPR table.


In [ ]:
def _read_csv_if_exists(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()


def _resolve_runs_root_for_tpr_fpr_tables():
    if "RUNS_ROOT" in globals():
        return Path(RUNS_ROOT)

    project_root = PROJECT_ROOT if "PROJECT_ROOT" in globals() else find_project_root()

    candidates = [
        project_root / "Experiment_1" / "training_runs_final_fixed" / "runs",
        project_root / "Experiment_2" / "TR_fixed_skin" / "runs",
        project_root / "Experiment_2" / "TR_fixed" / "runs",
        project_root / "Experiment_3" / "TR_fixed" / "runs",
    ]
    existing = [path for path in candidates if path.exists()]
    if len(existing) == 1:
        return existing[0]
    if len(existing) > 1:
        print("RUNS_ROOT was not defined. Using first existing candidate:", existing[0])
        return existing[0]
    raise NameError("RUNS_ROOT is not defined and no default runs directory was found.")


def _load_all_average_results():
    rows = []
    runs_root = _resolve_runs_root_for_tpr_fpr_tables()
    for avg_path in sorted(runs_root.rglob("averages.csv")):
        df = _read_csv_if_exists(avg_path)
        if df.empty:
            continue
        df = df.copy()
        df["averages_path"] = str(avg_path)
        rows.append(df)
    return pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()


def _numeric_columns(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _select_best_rows_by_validation_objective(avg_df):
    if avg_df.empty or "val_objective" not in avg_df.columns:
        return pd.DataFrame()

    avg_df = avg_df.copy()
    avg_df = _numeric_columns(
        avg_df,
        [
            "lr", "batch_size", "beta", "val_objective", "val_auc", "test_objective", "test_auc",
            "val_accuracy", "test_accuracy", "val_avg_tpr", "test_avg_tpr", "val_avg_fpr", "test_avg_fpr",
            "val_G", "test_G", "val_eo_gap_skin", "test_eo_gap_skin", "val_eo_gap_sex", "test_eo_gap_sex",
            "val_eo_gap_age", "test_eo_gap_age",
        ],
    )

    group_cols = [col for col in ["task", "sensor"] if col in avg_df.columns]
    if not group_cols:
        return avg_df.sort_values(["val_objective", "val_auc"], ascending=[False, False]).head(1)

    selected = []
    sort_cols = [col for col in ["val_objective", "val_auc", "test_objective", "test_auc"] if col in avg_df.columns]
    for _, group_df in avg_df.groupby(group_cols, sort=False):
        selected.append(group_df.sort_values(sort_cols, ascending=[False] * len(sort_cols)).iloc[0])
    return pd.DataFrame(selected).reset_index(drop=True)


def _build_best_summary_with_avg_tpr_fpr():
    avg_df = _load_all_average_results()
    best_df = _select_best_rows_by_validation_objective(avg_df)
    if best_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in best_df.iterrows():
        for split, label in [("val", "Validation"), ("test", "Test")]:
            rows.append({
                "Task": row.get("task", ""),
                "Sensor": row.get("sensor", ""),
                "Model": row.get("model", ""),
                "HP Tag": row.get("hp_tag", ""),
                "LR": row.get("lr", np.nan),
                "Batch Size": row.get("batch_size", np.nan),
                "Beta": row.get("beta", np.nan),
                "Metric": label,
                "Accuracy": row.get(f"{split}_accuracy", np.nan),
                "AUC": row.get(f"{split}_auc", np.nan),
                "Average TPR": row.get(f"{split}_avg_tpr", np.nan),
                "Average FPR": row.get(f"{split}_avg_fpr", np.nan),
                "EO Gap (Skin)": row.get(f"{split}_eo_gap_skin", np.nan),
                "EO Gap (Sex)": row.get(f"{split}_eo_gap_sex", np.nan),
                "EO Gap (Age)": row.get(f"{split}_eo_gap_age", np.nan),
                "G": row.get(f"{split}_G", np.nan),
                "Objective": row.get(f"{split}_objective", np.nan),
            })

    out_df = pd.DataFrame(rows)
    if out_df["Beta"].isna().all():
        out_df = out_df.drop(columns=["Beta"])
    return out_df


def _class_label_from_suffix(suffix):
    if suffix in {"benign", "malignant"}:
        return suffix.capitalize()
    return suffix.upper()


def _build_best_per_class_tpr_fpr_table():
    avg_df = _load_all_average_results()
    best_df = _select_best_rows_by_validation_objective(avg_df)
    if best_df.empty:
        return pd.DataFrame()

    rows = []
    for _, row in best_df.iterrows():
        for split, label in [("val", "Validation"), ("test", "Test")]:
            prefix = f"{split}_tpr_"
            class_suffixes = sorted([col[len(prefix):] for col in row.index if col.startswith(prefix)])
            for suffix in class_suffixes:
                rows.append({
                    "Task": row.get("task", ""),
                    "Sensor": row.get("sensor", ""),
                    "Model": row.get("model", ""),
                    "HP Tag": row.get("hp_tag", ""),
                    "Beta": row.get("beta", np.nan),
                    "Metric": label,
                    "Class": _class_label_from_suffix(suffix),
                    "TPR": row.get(f"{split}_tpr_{suffix}", np.nan),
                    "FPR": row.get(f"{split}_fpr_{suffix}", np.nan),
                })

    out_df = pd.DataFrame(rows)
    if not out_df.empty and out_df["Beta"].isna().all():
        out_df = out_df.drop(columns=["Beta"])
    return out_df


best_summary_with_avg_tpr_fpr = _build_best_summary_with_avg_tpr_fpr()
print("Best configurations selected by validation Objective, including average TPR/FPR")
display(best_summary_with_avg_tpr_fpr)

best_per_class_tpr_fpr_table = _build_best_per_class_tpr_fpr_table()
print("Per-class TPR/FPR for the best configurations selected by validation Objective")
display(best_per_class_tpr_fpr_table)
